# Milestone 1 - Crawling

Báo cáo thu thập corpus arXiv: khám phá các bài báo được giao, tải mã nguồn LaTeX của **mọi version**, crawl metadata và reference. Mọi bảng và biểu đồ trong trang này được **sinh lúc compile book** từ file `data/benchmark_m1.json` do `run_m1.py` xuất ra, không có ảnh chụp tĩnh.

## 1. Tổng quan quy trình

```text
ids.txt ──► Discovery (arXiv API) ──► với mỗi batch 25 bài:
                                        ├─ Semantic Scholar (1 request batch) ─► references
                                        └─ với mỗi version vN:
                                             tải export.arxiv.org/src/{id}vN ─► giữ .tex/.bib ─► metadata.json
```

| Bước | Công cụ | Cách làm |
|---|---|---|
| Đọc danh sách ID | Python | Bỏ dòng trống, dòng chú thích, hậu tố `vN` và ID trùng. |
| Entry discovery | arXiv API qua `arxiv.py` (`Client.results`) | Lượt 1: tra `id_list` theo ID gốc, hậu tố version của kết quả cho biết bài có bao nhiêu version. Lượt 2: tra từng ID có hậu tố `vN` để lấy ngày của từng version. Mỗi lần gọi cách nhau 3 giây, gom 50 ID mỗi request. |
| Tải source | `urllib` tới mirror `export.arxiv.org/src/{id}vN` | Mỗi version tải riêng. Dùng User-Agent mô tả và giãn cách tối thiểu 3 giây giữa hai lần tải (lớp `RateLimiter`). Gặp 429/5xx thì chờ rồi thử lại tối đa 4 lần. |
| Loại bỏ hình | `tarfile`, `gzip` | **Chỉ giải nén `.tex` và `.bib`**, nên file hình và file khác không bao giờ vào corpus; các lệnh `\includegraphics`, môi trường `figure` trong `.tex` vẫn giữ nguyên cho Milestone 2. Gói PDF-only hoặc không nhận dạng được thì ghi bản rỗng. |
| Reference | Semantic Scholar Graph API qua `requests` | Gọi endpoint batch (25 bài mỗi request) để giảm số lần gọi và tránh giới hạn tốc độ; giữ toàn bộ reference, trong đó reference có `externalIds.ArXiv` sẽ vào bảng `references`. |
| Đo hiệu năng | `psutil`, `time.perf_counter` (`PerfRecorder`) | Lấy mẫu RAM của cả cây tiến trình mỗi 0,5 giây; quét dung lượng đĩa ở luồng nền. |

Vì arXiv yêu cầu giãn cách 3 giây và mỗi version là một lần tải độc lập, **tải source là phần không thể song song hoá**; reference được gom batch nên không phải điểm nghẽn.

## 2. Cấu trúc dữ liệu thu được

Mỗi bài có một thư mục `<paper_key>` (ví dụ `2603.00580` thành `2603-00580`):

| File | Nội dung |
|---|---|
| `paper.json` | Metadata cấp bài: `arxiv_id`, `title`, `authors`, `submission_date`, `revised_dates`, `venue`, `n_versions`. Nguồn của bảng `papers`. |
| `vN/*.tex`, `vN/*.bib` | Mã nguồn đã bỏ hình. Version không có TeX vẫn có thư mục `vN/` rỗng. |
| `vN/metadata.json` | Metadata version: `source_kind`, `archive_bytes`, `bytes_before`, `bytes_after`. |
| `vN/references.json` | **Toàn bộ** reference từ Semantic Scholar (`ref_s2_id`, `ref_arxiv_id`, `ref_title`, `ref_authors`, `ref_submission_date`, `ref_venue`). Chỉ reference có arXiv ID vào bảng `references` cuối. |

ID cũ dạng `hep-th/9901001` được giữ nguyên trong dữ liệu, chỉ đổi `/` và `.` thành `-` khi đặt tên thư mục.

## 3. Định nghĩa các chỉ số

- **Bài thành công:** mọi version của bài đều tải và ghi được. Chỉ một version lỗi cũng làm cả bài thất bại.
- **Độ bao phủ version:** số version tải được trên tổng số version arXiv báo có. Chỉ số này phản ánh sát lượng dữ liệu thực thu được hơn tỉ lệ thành công theo bài.
- **Kích thước trước:** tổng dung lượng không nén mọi file trong gói source (kể cả hình). **Kích thước sau:** tổng dung lượng `.tex` và `.bib` giữ lại. Số liệu theo bài là cộng dồn các version có TeX.
- **Version PDF-only:** version không có `.tex`, được ghi bản rỗng và không tính vào thống kê kích thước.
- **Crawl reference thành công:** Semantic Scholar trả về danh sách reference cho bài (có thể rỗng).

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA = Path("data/benchmark_m1.json")
bench = json.loads(DATA.read_text(encoding="utf-8"))

# Đặt False khi data/benchmark_m1.json là kết quả chạy đầy đủ của nhóm
PROVISIONAL = True
# Ghi rõ môi trường đo của file benchmark này
MACHINE = "máy cá nhân (Windows), số liệu thử"

if PROVISIONAL:
    print("LƯU Ý: số liệu hiện là bản chạy thử trên một tập nhỏ, sẽ được thay bằng kết quả chạy đầy đủ.")
print(f"Môi trường đo: {MACHINE}")
print(f"Số bài trong file benchmark: {bench['n_papers_timed']}")

# Bảng theo từng bài, gộp từ các khóa paper_* của benchmark
papers = pd.DataFrame({"ok": pd.Series(bench["paper_results"]),
                       "seconds": pd.Series(bench["paper_times_s"])})
sz = pd.DataFrame(bench["paper_sizes_bytes"]).T
rf = pd.DataFrame(bench["paper_references"]).T
papers = papers.join(sz).join(rf[["total", "with_arxiv"]].astype(int))
papers.index.name = "arxiv_id"
papers = papers.reset_index()

## 4. Kết quả crawl

In [ ]:
cov = bench["coverage"]
n_papers = len(papers)
summary = pd.DataFrame([
    ("Số bài được giao", n_papers),
    ("Số bài crawl thành công", bench["n_ok"]),
    ("Số bài thất bại", bench["n_failed"]),
    ("Tỉ lệ thành công (theo bài)", f"{bench['success_rate']:.1%}"),
    ("Số version kỳ vọng", cov["expected_versions"]),
    ("Số version tải được", cov["crawled_versions"]),
    ("Số version lỗi", cov["failed_versions"]),
    ("Độ bao phủ version", f"{cov['version_coverage']:.1%}"),
    ("Số bài không tìm thấy trên arXiv API", cov.get("not_found_papers", 0)),
    ("Version có TeX", bench["n_tex_versions"]),
    ("Version chỉ có PDF (bản rỗng)", bench["n_pdf_only_versions"]),
], columns=["Chỉ số", "Giá trị"])
summary

Tỉ lệ thành công theo bài luôn thấp hơn độ bao phủ version, vì một version lỗi làm hỏng cả bài. Trong các lần chạy thử, lỗi chủ yếu là `IncompleteRead` khi tải file lớn (kết nối bị ngắt giữa chừng), đây là lỗi tạm thời nên cần thử lại thay vì bỏ qua.

In [ ]:
labels = papers["ok"].map({True: "Thành công", False: "Thất bại"}).value_counts()
px.pie(values=labels.values, names=labels.index, hole=0.4,
       title="Kết quả crawl theo bài").show()

failed = papers.loc[~papers["ok"], ["arxiv_id", "seconds"]].sort_values("seconds", ascending=False)
print(f"Số bài thất bại: {len(failed)}")
failed.head(20)

In [ ]:
# Phân loại lỗi: chỉ hiển thị khi nhóm có xuất file data/failures_m1.csv (cột: id, loai_loi)
FAIL = Path("data/failures_m1.csv")
if FAIL.exists():
    f = pd.read_csv(FAIL)
    px.bar(f["loai_loi"].value_counts().rename_axis("loại lỗi").reset_index(name="số lần"),
           x="loại lỗi", y="số lần", title="Phân loại lỗi tải source").show()
else:
    print("Chưa có data/failures_m1.csv, bỏ qua phân loại lỗi.")

## 5. Kích thước dữ liệu trước và sau khi bỏ hình

In [ ]:
size_tbl = pd.DataFrame([
    ("Kích thước TB mỗi bài, trước (byte)", f"{bench['avg_paper_size_before_bytes']:,}"),
    ("Kích thước TB mỗi bài, sau (byte)", f"{bench['avg_paper_size_after_bytes']:,}"),
    ("Kích thước TB mỗi version, trước (byte)", f"{bench['avg_version_size_before_bytes']:,}"),
    ("Kích thước TB mỗi version, sau (byte)", f"{bench['avg_version_size_after_bytes']:,}"),
    ("Mức giảm của kích thước TB mỗi bài",
     f"{100*(1-bench['avg_paper_size_after_bytes']/bench['avg_paper_size_before_bytes']):.1f}%"),
], columns=["Chỉ số", "Giá trị"])
size_tbl

In [ ]:
t = papers.dropna(subset=["before"]).copy()
t["before"] = t["before"].astype(float); t["after"] = t["after"].astype(float)
t["giảm (%)"] = 100 * (1 - t["after"] / t["before"])

stat = t[["before", "after"]].describe(percentiles=[.5, .9, .99]).T.round(0)
stat.index = ["Trước (byte)", "Sau (byte)"]
stat

In [ ]:
long = t.melt(id_vars="arxiv_id", value_vars=["before", "after"], var_name="loại", value_name="byte")
long["loại"] = long["loại"].map({"before": "Trước (cả hình)", "after": "Sau (.tex/.bib)"})
px.histogram(long, x="byte", color="loại", log_x=True, nbins=40, barmode="overlay", opacity=.7,
             title="Phân bố kích thước mỗi bài (thang log)").show()

px.scatter(t, x="before", y="after", log_x=True, log_y=True, hover_name="arxiv_id",
           title="Kích thước sau và trước khi bỏ hình (mỗi điểm là một bài)",
           labels={"before": "Trước (byte)", "after": "Sau (byte)"}).show()

px.histogram(t, x="giảm (%)", nbins=25, title="Phân bố mức giảm dung lượng sau khi bỏ hình").show()

Phần lớn dung lượng của một gói source là hình và file phụ, nên việc chỉ giữ `.tex`/`.bib` giảm dung lượng rất mạnh. Một số bài gần như không giảm vì gói source vốn chỉ chứa mã LaTeX. Bài có kích thước đuôi lớn nhất là các bài kèm nhiều hình.

In [ ]:
t.nlargest(10, "before")[["arxiv_id", "before", "after", "giảm (%)", "tex_versions"]].round(1)

## 6. Số version mỗi bài

In [ ]:
v = t["tex_versions"].astype(int).value_counts().sort_index().rename_axis("số version TeX").reset_index(name="số bài")
px.bar(v, x="số version TeX", y="số bài", title="Phân bố số version có TeX mỗi bài").show()
v

Chỉ tính các bài có ít nhất một version TeX. Bài càng nhiều version càng tốn thời gian tải, vì mỗi version là một lần tải riêng cách nhau 3 giây.

## 7. Reference

In [ ]:
ref_tbl = pd.DataFrame([
    ("Tỉ lệ crawl reference thành công", f"{bench['ref_success_rate']:.1%}"),
    ("Số reference TB mỗi bài", bench["avg_refs_per_paper"]),
    ("Số reference có arXiv ID TB mỗi bài", bench["avg_arxiv_refs_per_paper"]),
    ("Tổng số reference crawl được", int(papers["total"].sum())),
    ("Trong đó có arXiv ID (vào bảng references)", int(papers["with_arxiv"].sum())),
    ("Tỉ lệ reference có arXiv ID (gộp)", f"{papers['with_arxiv'].sum()/papers['total'].sum():.1%}"),
    ("Số bài có 0 reference", int((papers["total"] == 0).sum())),
], columns=["Chỉ số", "Giá trị"])
ref_tbl

In [ ]:
px.histogram(papers, x="total", nbins=40, title="Phân bố số reference crawl được mỗi bài",
             labels={"total": "Số reference"}).show()

r = papers[papers["total"] > 0].copy()
r["tỉ lệ có arXiv ID"] = r["with_arxiv"] / r["total"]
px.histogram(r, x="tỉ lệ có arXiv ID", nbins=20,
             title="Tỉ lệ reference có arXiv ID mỗi bài (bài có reference)").show()

Chỉ reference có arXiv ID mới vào bảng `references` cuối cùng, đây là tập đích để ghép ở Milestone 3, nên tỉ lệ này quyết định kích thước tập ứng viên. Các bài có 0 reference thường do Semantic Scholar chưa index bài mới (độ trễ 1 đến 7 ngày), và được tính là crawl thành công vì API trả về hợp lệ.

## 8. Hiệu năng: thời gian

In [ ]:
times = papers["seconds"]
time_tbl = pd.DataFrame([
    ("Tổng thời gian end-to-end (giây)", bench["total_wall_s"]),
    ("Entry discovery (giây)", bench["discovery_s"]),
    ("Giai đoạn crawl (giây)", bench["stages_s"]["crawl"]),
    ("Thời gian TB mỗi bài (giây)", bench["avg_paper_s"]),
    ("Trung vị mỗi bài (giây)", round(times.median(), 2)),
    ("Phân vị 90 (giây)", round(times.quantile(.9), 2)),
    ("Lâu nhất của một bài (giây)", bench["max_paper_s"]),
], columns=["Chỉ số", "Giá trị"])
time_tbl

In [ ]:
px.histogram(papers, x="seconds", nbins=40, log_y=True,
             title="Phân bố thời gian xử lý mỗi bài", labels={"seconds": "giây"}).show()

s = papers.dropna(subset=["before"]).copy()
s["before_mb"] = s["before"].astype(float) / 1e6
px.scatter(s, x="before_mb", y="seconds", color=s["tex_versions"].astype(int).astype(str),
           hover_name="arxiv_id", labels={"before_mb": "Kích thước gói source (MB)", "seconds": "giây", "color": "số version"},
           title="Thời gian xử lý theo kích thước gói source và số version").show()

Thời gian mỗi bài gồm phần chia đều của request reference batch và thời gian tải từng version. Đuôi dài của phân bố đến từ bài nhiều version và gói source lớn; mức sàn khoảng vài giây đến từ giãn cách 3 giây bắt buộc.

## 9. Hiệu năng: bộ nhớ

In [ ]:
mem_tbl = pd.DataFrame([
    ("RAM tối đa (MB)", bench["peak_ram_mb"]),
    ("RAM trung bình (MB)", bench["avg_ram_mb"]),
    ("Số mẫu RAM", bench["ram_samples"]),
    ("Đĩa tối đa (MB)", bench["peak_disk_mb"]),
    ("Dung lượng output cuối (MB)", bench["final_output_mb"]),
    ("Dung lượng đĩa", "ước tính" if bench["disk_estimated"] else "đo thật"),
], columns=["Chỉ số", "Giá trị"])
mem_tbl

RAM đo trên cả cây tiến trình, lấy mẫu mỗi 0,5 giây. Đỉnh đĩa gồm cả gói source tạm trước khi giải nén. Số liệu chính thức sẽ đo trên Google Colab chế độ CPU.

## 10. Nhận xét và hạn chế

- Thời gian crawl tỉ lệ thuận với tổng số version cần tải, vì arXiv bắt buộc giãn cách 3 giây; không thể rút ngắn bằng song song hoá.
- Một số bài thất bại do lỗi mạng tạm thời khi tải file lớn. Cần retry cho lỗi đọc dở để tăng độ bao phủ version.
- Reference lấy từ Semantic Scholar chứ không từ `.bib` của bài, nên chất lượng phụ thuộc mức độ index của dịch vụ này; khoảng một nửa số reference có arXiv ID.
- Dung lượng giữ lại rất nhỏ so với gói gốc, nên toàn bộ corpus lưu được trên đĩa của Colab.
- Các số liệu hiệu năng đo trên một lượt chạy, chưa lặp lại nhiều lần.